# Stock Predictor: findings explorer

This notebook loads the results saved by `run_analysis.py` (20 tickers, Yahoo Finance daily closes, 2021-10-04 to 2026-10-01) and lets you look at the per-ticker detail behind [`docs/FINDINGS.md`](../docs/FINDINGS.md). It does not retrain anything, so it runs in seconds. To regenerate the results from scratch run `python analysis/run_analysis.py` (about 3 minutes, needs internet).

In [1]:
import json
from pathlib import Path
import pandas as pd

R = Path('results') if Path('results').exists() else Path('analysis/results')
pd.options.display.float_format = '{:.3f}'.format
meta = json.loads((R / 'summary.json').read_text())
print('data:', meta['data'])
print('window:', min(meta['first_date_5y'].values()), 'to', meta['end_date'])
print('tickers:', ', '.join(meta['tickers']))

data: Yahoo Finance via yfinance, adjusted daily closes
window: 2021-10-04 to 2026-10-01
tickers: SPY, QQQ, AAPL, MSFT, NVDA, TSLA, AMZN, GOOGL, META, JPM, KO, XOM, PFE, JNJ, PG, WMT, UNH, XLE, TLT, GLD


## 1. Return forecasts versus "price stays flat"

![point forecasts](figures/1_point_forecasts_vs_flat.png)

Skill > 0 means a smaller error than flat. Each row below is one ticker at a 20-day horizon.

In [2]:
p = pd.read_csv(R / 'point_models.csv')
wide = p[p.horizon == 20].pivot(index='ticker', columns='model', values='skill')[['drift', 'ewma', 'ridge_ar', 'gbm']]
wide.assign(best=wide.idxmax(axis=1)).sort_values('gbm')

model,drift,ewma,ridge_ar,gbm,best
ticker,,,,,
META,0.007,-0.077,-0.036,-0.387,drift
TLT,-0.083,-0.089,-0.219,-0.272,drift
PFE,-0.070,-0.086,-0.087,-0.163,drift
NVDA,0.038,-0.037,-0.022,-0.123,drift
SPY,0.046,-0.015,-0.009,-0.120,drift
WMT,0.024,-0.005,0.033,-0.119,ridge_ar
TSLA,-0.021,-0.079,-0.026,-0.109,drift
MSFT,-0.003,-0.069,-0.008,-0.097,drift
GOOGL,0.012,-0.025,-0.012,-0.091,drift


Direction: how often was the model's up/down call right, compared with always saying "up"? (gradient boosting, 20 days)

In [3]:
g = p[(p.model == 'gbm') & (p.horizon == 20)].set_index('ticker')[['hit_rate', 'up_rate', 'skill', 'verdict']]
g.assign(beats_always_up=g.hit_rate > g.up_rate)

,hit_rate,up_rate,skill,verdict,beats_always_up
ticker,,,,,
SPY,0.560,0.722,-0.120,not_better,False
QQQ,0.604,0.681,-0.069,not_better,False
AAPL,0.536,0.649,-0.070,not_better,False
MSFT,0.440,0.511,-0.097,not_better,False
NVDA,0.609,0.616,-0.123,not_better,False
TSLA,0.497,0.550,-0.109,not_better,False
AMZN,0.516,0.577,-0.068,not_better,False
GOOGL,0.519,0.602,-0.091,not_better,False
META,0.378,0.550,-0.387,not_better,False


## 2. Leakage: the same model scored three ways

![leakage](figures/2_leakage_demo.png)

In [4]:
leak = pd.read_csv(R / 'leakage.csv').set_index('ticker')
leak.agg(['min', 'median', 'max'])

,shuffled k-fold (leaky),"walk-forward, no embargo",walk-forward + embargo (used)
min,0.066,-0.365,-0.387
median,0.106,-0.082,-0.086
max,0.202,0.019,0.016


## 3. Volatility

![volatility models](figures/3_volatility_models.png)

GJR-GARCH versus the EWMA headline, per ticker (skill_vs_ewma > 0 means GARCH had the smaller error; verdict uses the 90% bootstrap range).

In [5]:
v = pd.read_csv(R / 'vol_models.csv')
gv = v[v.model == 'garch'].pivot(index='ticker', columns='horizon', values='skill_vs_ewma')
gv.columns = [f'{h}d' for h in gv.columns]
gv

,5d,20d,60d,120d
ticker,,,,
AAPL,0.032,0.110,0.247,0.409
AMZN,0.028,0.135,0.282,0.318
GLD,0.086,0.059,-0.003,-0.073
GOOGL,0.006,0.121,0.325,0.492
JNJ,0.064,0.165,0.237,0.446
JPM,0.054,0.131,0.307,0.455
KO,0.046,0.119,0.174,0.351
META,-0.245,-0.204,-0.145,-0.007
MSFT,0.046,0.159,0.244,0.384


In [6]:
v[(v.model == 'garch') & (v.horizon == 20)].set_index('ticker')[['skill', 'skill_vs_ewma', 'ci_vs_ewma_lo', 'ci_vs_ewma_hi', 'verdict_vs_ewma']]

,skill,skill_vs_ewma,ci_vs_ewma_lo,ci_vs_ewma_hi,verdict_vs_ewma
ticker,,,,,
SPY,0.142,0.137,0.059,0.213,better
QQQ,0.083,0.097,-0.013,0.179,inconclusive
AAPL,0.131,0.110,0.006,0.202,better
MSFT,0.213,0.159,0.061,0.229,better
NVDA,-0.338,-0.296,-0.632,-0.064,not_better
TSLA,0.091,-0.021,-0.192,0.094,not_better
AMZN,0.200,0.135,0.059,0.220,better
GOOGL,0.206,0.121,-0.007,0.223,inconclusive
META,-0.082,-0.204,-0.417,-0.015,not_better


### SPY case study

![SPY volatility](figures/4_volatility_case_study.png)

In [7]:
cs = pd.read_csv(R / 'vol_case_study.csv', index_col=0, parse_dates=True)
print(len(cs), 'out-of-sample rows,', cs.index.min().date(), 'to', cs.index.max().date())
cs.agg(['mean', 'min', 'max'])

584 out-of-sample rows, 2024-05-06 to 2026-09-02


,actual,ewma,garch
mean,0.141,0.147,0.156
min,0.061,0.074,0.099
max,0.524,0.518,0.447


## 4. Is the 80% range 80%?

![coverage](figures/5_interval_coverage.png)

In [8]:
c = pd.read_csv(R / 'conformal_vs_normal.csv')
c[c.horizon == 20][['ticker', 'conformal_cov', 'ci_lo', 'ci_hi', 'normal_cov', 'multiplier', 'n_independent', 'used']].set_index('ticker')

,conformal_cov,ci_lo,ci_hi,normal_cov,multiplier,n_independent,used
ticker,,,,,,,
SPY,0.800,0.637,0.901,0.816,1.221,23,True
QQQ,0.814,0.654,0.911,0.846,1.204,23,True
AAPL,0.812,0.651,0.909,0.734,1.484,23,True
MSFT,0.719,0.551,0.843,0.745,1.401,23,True
NVDA,0.838,0.680,0.926,0.810,1.300,23,True
TSLA,0.861,0.707,0.941,0.800,1.335,23,True
AMZN,0.793,0.630,0.896,0.772,1.387,23,True
GOOGL,0.728,0.560,0.849,0.698,1.529,23,True
META,0.817,0.656,0.912,0.673,1.829,23,True


In [9]:
c.groupby('horizon').agg(used=('used', 'sum'), conformal_median=('conformal_cov', 'median'), normal_median=('normal_cov', 'median'),
                       independent_periods=('n_independent', 'median'))

,used,conformal_median,normal_median,independent_periods
horizon,,,,
5,20,0.814,0.792,99.000
20,20,0.813,0.784,23.000
60,0,0.773,0.778,6.000
120,20,0.810,0.768,16.000
180,20,0.784,0.701,9.000
256,0,0.865,0.732,5.000


## 5. Want to re-run it?

Uncomment to regenerate everything (about 3 minutes; downloads prices from Yahoo Finance):

In [10]:
# import subprocess, sys
# subprocess.run([sys.executable, 'run_analysis.py'], check=True)